In [4]:
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_community.retrievers import WikipediaRetriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

load_dotenv()

True

# Wikipedia Retriever

In [4]:
retriever= WikipediaRetriever(top_k_results=2, lang='en')

In [9]:
query= "World War 1"
docs= retriever.invoke(query)

In [10]:
for i, doc in enumerate(docs):
    print(f'document: {i+1}')
    print(f"\n{doc.page_content}\n")

document: 1

World War I, or the First World War (28 July 1914 – 11 November 1918), also known as the Great War, was a global conflict between two coalitions: the Allies (or Entente) and the Central Powers. Major areas of conflict included Europe and the Middle East, as well as parts of Africa and the Asia-Pacific. The war saw important developments in weaponry including tanks, aircraft, artillery, machine guns, and chemical weapons. One of the deadliest conflicts in history, it resulted in an estimated 30 million military casualties, and 8 million civilian deaths from war-related causes and genocide. The movement of large numbers of people was a major factor in the deadly Spanish flu pandemic. 
The causes of World War I included the rise of the German Empire and decline of the Ottoman Empire, which disturbed the long-standing balance of power in Europe, the exacerbation of imperial rivalries, and an arms race between the great powers. Growing tensions in the Balkans reached a breaking

In [11]:
docs

[Document(metadata={'title': 'World War I', 'summary': "World War I, or the First World War (28 July 1914 – 11 November 1918), also known as the Great War, was a global conflict between two coalitions: the Allies (or Entente) and the Central Powers. Major areas of conflict included Europe and the Middle East, as well as parts of Africa and the Asia-Pacific. The war saw important developments in weaponry including tanks, aircraft, artillery, machine guns, and chemical weapons. One of the deadliest conflicts in history, it resulted in an estimated 30 million military casualties, and 8 million civilian deaths from war-related causes and genocide. The movement of large numbers of people was a major factor in the deadly Spanish flu pandemic. \nThe causes of World War I included the rise of the German Empire and decline of the Ottoman Empire, which disturbed the long-standing balance of power in Europe, the exacerbation of imperial rivalries, and an arms race between the great powers. Growin

# Vector Store Retriever

In [13]:
documents= [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM search."),
    Document(page_content="Embeddings convert text into high dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models.")
]

embedding_model= MistralAIEmbeddings()

In [14]:
vectorstore = Chroma.from_documents(documents, embedding_model, persist_directory="chroma_db")

retriever= vectorstore.as_retriever(search_kwargs={"k": 2})

In [15]:
query= "What is Chroma used for?"
results= retriever.invoke(query)

for i, docs in enumerate(results):
    print(f"document: {i+1}")
    print(f"\n{docs.page_content}\n")

document: 1

Chroma is a vector database optimized for LLM search.

document: 2

Embeddings convert text into high dimensional vectors.



# Maximal Marginal Relevance (MMR)

In [22]:
docs = [

    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM search."),
    Document(page_content="Embeddings convert text into high dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),

    Document(page_content="LangChain is a framework for building applications powered by large language models."),
    Document(page_content="LangChain provides tools for prompt templates, chains, agents, and memory management."),
    Document(page_content="Developers use LangChain to connect LLMs with external data sources and APIs."),
    Document(page_content="LangChain simplifies building chatbots, retrieval augmented generation systems, and AI assistants."),
    Document(page_content="LangChain integrates with vector databases like FAISS and Chroma for semantic search."),

    Document(page_content="FAISS is a library developed by Meta for efficient similarity search over vector embeddings."),
    Document(page_content="FAISS enables fast nearest neighbor search in large vector datasets."),
    Document(page_content="Many machine learning systems use FAISS to perform semantic search and clustering."),
    Document(page_content="FAISS supports indexing techniques that allow scalable vector retrieval."),
    Document(page_content="FAISS is widely used in recommendation systems and embedding-based search applications."),

    Document(page_content="Chroma is an open source vector database designed for storing and searching embeddings."),
    Document(page_content="Chroma works well with LangChain for building retrieval based AI applications."),
    Document(page_content="Developers use Chroma to store document embeddings and perform semantic similarity queries."),
    Document(page_content="Chroma provides persistent storage and fast retrieval for vector based search."),
    Document(page_content="Chroma is commonly used in retrieval augmented generation pipelines.")
]

In [23]:
embedding_model= MistralAIEmbeddings()

vectorstore= FAISS.from_documents(
    documents= docs,
    embedding= embedding_model
)

In [28]:
retriever= vectorstore.as_retriever(
    search_type= 'mmr',
    search_kwargs= {'k': 3, "lambda_mult": 0.5} #k= no. of top results, lambda_mult= diversity of results
)

In [29]:
query= "What is LangChain?"

results= retriever.invoke(query)

for i, docs in enumerate(results):
    print(f"document: {i+1}")
    print(f"\n{docs.page_content}\n")
    print("-----------")
    print()

document: 1

LangChain is a framework for building applications powered by large language models.

-----------

document: 2

Chroma provides persistent storage and fast retrieval for vector based search.

-----------

document: 3

LangChain integrates with vector databases like FAISS and Chroma for semantic search.

-----------



# Multi-Query Retriever

In [30]:
all_docs = [

    # -------- Health Related Docs (H) --------
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),

    # Additional Health Docs
    Document(page_content="Daily exercise strengthens the immune system and improves overall well-being.", metadata={"source": "H6"}),
    Document(page_content="A balanced diet with proteins, vitamins, and minerals supports healthy body function.", metadata={"source": "H7"}),
    Document(page_content="Meditation can reduce stress and improve concentration over time.", metadata={"source": "H8"}),
    Document(page_content="Quality sleep enhances memory consolidation and cognitive performance.", metadata={"source": "H9"}),
    Document(page_content="Regular hydration supports digestion and nutrient absorption.", metadata={"source": "H10"}),

    # -------- Informational / Mixed Knowledge Docs (I) --------
    Document(page_content="Solar energy systems in modern homes help reduce electricity costs and carbon emissions.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular programming language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight into chemical energy.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and attracted global attention.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and contain extremely strong gravitational forces.", metadata={"source": "I5"}),

    # Additional Informational Docs
    Document(page_content="Wind energy is generated by converting air flow into mechanical power using turbines.", metadata={"source": "I6"}),
    Document(page_content="Machine learning allows computers to learn patterns from data without explicit programming.", metadata={"source": "I7"}),
    Document(page_content="The internet connects millions of computers worldwide through standardized communication protocols.", metadata={"source": "I8"}),
    Document(page_content="Satellites orbit the Earth to support communication, navigation, and weather monitoring.", metadata={"source": "I9"}),
    Document(page_content="Artificial intelligence systems can analyze large datasets to generate useful insights.", metadata={"source": "I10"}),
]

In [ ]:
embedding_model= MistralAIEmbeddings()

vectorstore= FAISS.from_documents(
    documents= all_docs,
    embedding= embedding_model
)

In [42]:
similarity_retriever= vectorstore.as_retriever(search_type= 'similarity', search_kwargs= {'k': 3})

In [43]:
multiquery_retriever= MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs= {'k': 3}),
    llm=ChatMistralAI(model="mistral-medium-2508")
)

In [ ]:
Query= 'How to improve balance?'
similarity_results= similarity_retriever.invoke(Query)
multiquery_results= multiquery_retriever.invoke(Query)

print("MultiQuery results")
for i, docs in enumerate(multiquery_results):
    print(f"document: {i+1}")
    print(f"{docs.page_content}")
    print("-----------")
    print()

print("Similarity results")
for i, docs in enumerate(similarity_results):
    print(f"document: {i+1}")
    print(f"{docs.page_content}")
    print("-----------")
    print()

# Contextual Compression Retriever

In [2]:
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells."""
    ), metadata={"source": "Doc4"}),
]

In [3]:
embedding_model= MistralAIEmbeddings()

vectorstore= FAISS.from_documents(
    documents= docs,
    embedding= embedding_model
)

In [5]:
base_retriever= vectorstore.as_retriever(search_kwargs= {'k': 4})

llm= ChatMistralAI(model="mistral-medium-2508")

compressor= LLMChainExtractor.from_llm(llm)

compression_retriever= ContextualCompressionRetriever(
    base_retriever= base_retriever,
    base_compressor= compressor
)

In [6]:
query= 'What is photosynthesis?'

compressed_results= compression_retriever.invoke(query)

print("Compressed results")
for i, docs in enumerate(compressed_results):
    print(f"document: {i+1}")
    print(f"{docs.page_content}")
    print("-----------")
    print()
    print()
    print()

Compressed results
document: 1
"Photosynthesis is the process by which green plants convert sunlight into energy."
-----------



document: 2
Extracted relevant parts: **"Photosynthesis does not occur in animal cells."**
-----------



document: 3
The chlorophyll in plant cells captures sunlight during photosynthesis.
-----------



